In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

folder = r"C:\Users\Dana\OneDrive\Documents\Big 10 Meets"
folder = folder.strip('"')          # handles the quotes from "Copy as path"
data_dir = Path(folder)

files = sorted(data_dir.glob("*.csv"))
print("Loaded files:", [f.name for f in files])

df_all = pd.concat(
    [pd.read_csv(f).assign(meet_id=f.stem, conference="Big Ten") for f in files],
    ignore_index=True
)

df = df_all  # keep your later cells working

print("Rows:", len(df_all), "| Meets:", df_all["meet_id"].nunique())
df_all[["meet_id","Final Place","Athlete","POINTS"]].head()

In [ ]:
def time_to_seconds(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s.upper() in {"NA","DNS","DNF","NH","NM","-","--",""}:
        return np.nan
    m, sec = s.split(":")
    return int(m) * 60 + float(sec)

df["1500m_sec"] = df["1500m"].apply(time_to_seconds)

df[["meet_id", "Athlete", "1500m", "1500m_sec"]].head()

In [ ]:
df_rank = df.copy()

# --- FIX: coerce PV to numeric (NH/DNS/etc -> NaN) ---
bad = {"NA","DNS","DNF","NH","NM","NP","-","--",""}
pv = df_rank["PV"].astype(str).str.strip()
pv = pv.where(~pv.str.upper().isin(bad), np.nan)
pv = pv.str.extract(r'([0-9]*\.?[0-9]+)')[0]   # pull numeric part if any
df_rank["PV"] = pd.to_numeric(pv, errors="coerce")
# ------------------------------------------------------

# helper: rank within each meet
g = df_rank.groupby("meet_id")

# track (lower time is better)
df_rank["rank_100m"]  = g["100m"].rank(ascending=True, method="min")
df_rank["rank_400m"]  = g["400m"].rank(ascending=True, method="min")
df_rank["rank_110H"]  = g["110mH"].rank(ascending=True, method="min")
df_rank["rank_1500m"] = g["1500m_sec"].rank(ascending=True, method="min")

# field (higher mark is better)
df_rank["rank_LJ"] = g["LJ"].rank(ascending=False, method="min")
df_rank["rank_SP"] = g["SP"].rank(ascending=False, method="min")
df_rank["rank_HJ"] = g["HJ"].rank(ascending=False, method="min")
df_rank["rank_DT"] = g["DT"].rank(ascending=False, method="min")
df_rank["rank_PV"] = g["PV"].rank(ascending=False, method="min")
df_rank["rank_JT"] = g["JT"].rank(ascending=False, method="min")

df_rank[["meet_id","Final Place","Athlete",
         "rank_100m","rank_LJ","rank_SP","rank_HJ","rank_400m","rank_110H","rank_DT","rank_PV","rank_JT","rank_1500m"]].head()


For Spearman's test, all that is needed is the event ranking, not the specifics of the result.

In [ ]:
rank_cols = [c for c in df_rank.columns if c.startswith("rank_")]

spearman_rank = (
    df_rank[rank_cols]
    .corrwith(df_rank["Final Place"], method="spearman")
    .sort_values(ascending=False)
    .rename("spearman_rho")
    .reset_index()
    .rename(columns={"index": "event_rank"})
)

spearman_rank

In [ ]:
df_z = df.copy()

# --- FIX: coerce PV to numeric (NH/DNS/etc -> NaN) ---
bad = {"NA","DNS","DNF","NH","NM","NP","-","--",""}
pv = df_z["PV"].astype(str).str.strip()
pv = pv.where(~pv.str.upper().isin(bad), np.nan)
pv = pv.str.extract(r'([0-9]*\.?[0-9]+)')[0]
df_z["PV"] = pd.to_numeric(pv, errors="coerce")
# ------------------------------------------------------

def zscore_within_group(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd != 0 else np.nan

g = df_z.groupby("meet_id")

# track (flip sign so higher=better), then z within meet
df_z["z_100m"]  = g["100m"].transform(lambda x: zscore_within_group(-x))
df_z["z_400m"]  = g["400m"].transform(lambda x: zscore_within_group(-x))
df_z["z_110H"]  = g["110mH"].transform(lambda x: zscore_within_group(-x))
df_z["z_1500m"] = g["1500m_sec"].transform(lambda x: zscore_within_group(-x))

# field z within meet
df_z["z_LJ"] = g["LJ"].transform(zscore_within_group)
df_z["z_SP"] = g["SP"].transform(zscore_within_group)
df_z["z_HJ"] = g["HJ"].transform(zscore_within_group)
df_z["z_DT"] = g["DT"].transform(zscore_within_group)
df_z["z_PV"] = g["PV"].transform(zscore_within_group)
df_z["z_JT"] = g["JT"].transform(zscore_within_group)

z_cols = [c for c in df_z.columns if c.startswith("z_")]

df_z[["meet_id","Final Place","Athlete"] + z_cols].head()


In [ ]:
pearson_z = (
    df_z[z_cols]
    .corrwith(-df_z["Final Place"], method="pearson")
    .sort_values(ascending=False)
    .rename("pearson_r")
    .reset_index()
    .rename(columns={"index": "event_z"})
)

pearson_z